<div style="
background:linear-gradient(135deg,#2563EB,#1D4ED8);
padding:24px 28px;
border-radius:18px;
color:white;
margin-bottom:18px;">

<h1 style="margin:0;font-size:34px;">
🌍 Global Resilience Index
</h1>

<p style="margin:8px 0 0 0;font-size:17px;opacity:.9;">
World Bank Data Analysis • 100 Countries • 6 Domains • 2000–2023
</p>

</div>

## 📖 Project Overview

This notebook analyzes and forecasts **global resilience** by measuring how countries perform across six critical domains that influence long-term sustainability, stability, and development.

### Domains Covered

- 🌐 **Digital Infrastructure** — Internet access and broadband connectivity.
- 💰 **Economic Fragility** — GDP growth and inflation.
- 🍽️ **Food Security** — Food import dependency and undernourishment.
- 🏥 **Healthcare** — Health expenditure, hospital beds, and physicians.
- 🏛️ **Political Stability** — World Bank governance indicators.
- 🌱 **Climate & Energy** — Electricity access, renewable energy, CO₂ emissions, and energy consumption.

### 📊 Data Sources

- World Bank (15 Indicators)
- FAO Food Price Index

### 🎯 Project Objectives

- Build domain-level resilience scores.
- Compute an overall Global Resilience Score.
- Forecast resilience through **2030** using machine learning and regression models.
- Produce a final global ranking of countries based on predicted resilience.

In [1]:
#from IPython.display import Image, display

#display(Image(filename=r"C:\Users\ASUS\Documents\Depi\final.jpeg"))

In [2]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["font.family"] = "DejaVu Sans"


<div style="
background:linear-gradient(135deg,#06B6D4,#0284C7);
padding:20px 26px;
border-radius:18px;
color:white;
margin-bottom:18px;">

<h1 style="margin:0;font-size:30px;">
📥 Data Collection
</h1>

<p style="margin:8px 0 0 0;font-size:16px;opacity:.9;">
Loading 17 raw World Bank and FAO source files.
</p>

</div>

## 📂 1. Load Data

Prepare file paths and load all 17 source datasets into the analysis pipeline.

In [3]:
# ==============================================================================
# SECTION 1: FILE PATHS 
# ==============================================================================

path_fixed_broadband         = r"E:\DEPI Learn\Final_Data_EXCEL\Digital Infrastructure\Fixed broadband subscriptions.xlsx"
path_internet_users          = r"E:\DEPI Learn\Final_Data_EXCEL\Digital Infrastructure\Internet Users.xlsx"
path_gdp_growth              = r"E:\DEPI Learn\Final_Data_EXCEL\Economic Fragility\GDP Growth.xlsx"
path_inflation               = r"E:\DEPI Learn\Final_Data_EXCEL\Economic Fragility\Inflation.xlsx"
path_food_imports            = r"E:\DEPI Learn\Final_Data_EXCEL\Food\Food Imports % of merchandise imports.xlsx"
path_undernourishment        = r"E:\DEPI Learn\Final_Data_EXCEL\Food\Prevalence of Undernourishment.xlsx"
path_health_expenditure      = r"E:\DEPI Learn\Final_Data_EXCEL\Healthcare Capacity\Global Health Expenditure.xlsx"
path_hospitals               = r"E:\DEPI Learn\Final_Data_EXCEL\Healthcare Capacity\Hospitals.xlsx"
path_physicians              = r"E:\DEPI Learn\Final_Data_EXCEL\Healthcare Capacity\Physicians.xlsx"
path_political_stability     = r"E:\DEPI Learn\Final_Data_EXCEL\Political Stability\Political Stability.xlsx"
path_electricity_access      = r"E:\DEPI Learn\Final_Data_EXCEL\Climate & Energy\Access to electricity.xlsx"
path_clean_fuel              = r"E:\DEPI Learn\Final_Data_EXCEL\Climate & Energy\Clean Fuel Access.xlsx"
path_co2_emissions           = r"E:\DEPI Learn\Final_Data_EXCEL\Climate & Energy\CO2 emissions  .xlsx"
path_electricity_consumption = r"E:\DEPI Learn\Final_Data_EXCEL\Climate & Energy\Electricity consumption  .xlsx"
path_renewable_energy        = r"E:\DEPI Learn\Final_Data_EXCEL\Climate & Energy\Renewable energy  .xlsx"
path_fao_food_index          = r"E:\DEPI Learn\Final_Data_EXCEL\Food\FAO Food Price Index.xlsx"
path_government_debt         = r"E:\DEPI Learn\Final_Data_EXCEL\Economic Fragility\Government Debt.xlsx"
OUTPUT_DIR = r"E:\DEPI Learn\PythonProject\New folder"


<div style="
background:linear-gradient(135deg,#8B5CF6,#6D28D9);
padding:20px 26px;
border-radius:18px;
color:white;
margin-bottom:18px;">

<h1 style="margin:0;font-size:30px;">
⚙️ Data Understanding — Constants & Mappings
</h1>

<p style="margin:8px 0 0 0;font-size:16px;opacity:.9;">
Country list, domain definitions, region mapping, and indicator polarity.
</p>

</div>

## ⚙️ 2. Constants & Mappings

These constants define the **100 countries** included in the study, map each indicator to its corresponding resilience domain, assign countries to regions, and specify which indicators require polarity inversion (where higher values indicate lower resilience).

In [4]:
# The 100 countries included in this study
COUNTRY_CODES = {
    'GEO','MDA','ESP','CHE','GBR','HUN','MYS','RUS','CAN','ISL','NZL','TUR','KOR','MUS','THA',
    'JOR','KAZ','BLR','PHL','KGZ','AUS','BWA','COL','ARM','USA','CHL','LKA','MEX','IND','TUN',
    'AZE','BIH','UKR','CRI','ALB','ARE','URY','JAM','BRA','SLV','NOR','PER','AUT','BEL','CHN',
    'DEU','ISR','LVA','CZE','EST','FRA','IRL','ITA','MLT','NLD','PRT','PAK','IDN','FIN','LTU',
    'ROU','SVK','SVN','SWE','MNG','HRV','PAN','DNK','ECU','GRC','OMN','POL','SGP','BGD','BHR',
    'CYP','MAR','MKD','EGY','LUX','MOZ','BOL','KWT','ZMB','SAU','NPL','BFA','NIC','VNM','GTM',
    'KHM','RWA','PRY','TJK','TGO','ETH','SWZ','GHA','NAM','MDG','DOM','UZB'
}

# Exclude these name variants (same countries, different spelling in the data)
EXCLUDED_NAMES = {'Egypt, Arab Rep.', 'Israel'}

# Exclude future/incomplete years
EXCLUDED_YEARS = {2024, 2025}
EXCLUDED_FAO_YEARS = {2024, 2025, 2026}

# Indicators where HIGH value = BAD resilience → need to flip the score
INVERSE_INDICATORS = {
    'FP.CPI.TOTL.ZG',    # Inflation
    'NY.GDP.MKTP.KD.ZG', # GDP Growth (inverted in this model)
    'TM.VAL.FOOD.ZS.UN', # Food imports dependency
    'SN.ITK.DEFC.ZS',    # Undernourishment
    'EG.USE.ELEC.KH.PC', # Electricity consumption
}

# Map each indicator code to its domain
INDICATOR_DOMAIN = {
    'IT.NET.BBND.P2':       'Digital Infrastructure',
    'IT.NET.USER.ZS':       'Digital Infrastructure',
    'NY.GDP.MKTP.KD.ZG':    'Economic Fragility',
    'FP.CPI.TOTL.ZG':       'Economic Fragility',
    'TM.VAL.FOOD.ZS.UN':    'Food Security',
    'SN.ITK.DEFC.ZS':       'Food Security',
    'SH.XPD.CHEX.GD.ZS':   'Healthcare',
    'SH.MED.BEDS.ZS':       'Healthcare',
    'SH.MED.PHYS.ZS':       'Healthcare',
    'PV.EST':               'Political Stability',
    'EG.ELC.ACCS.ZS':       'Climate & Energy',
    'EG.CFT.ACCS.ZS':       'Climate & Energy',
    'EG.FEC.RNEW.ZS':       'Climate & Energy',
    'EG.USE.ELEC.KH.PC':    'Climate & Energy',
    'EN.GHG.CO2.PC.CE.AR5': 'Climate & Energy',
}

# Map country codes to regions
REGION_MAP = {
    **dict.fromkeys(['MAR','TUN','MOZ','ZMB','BWA','NAM','RWA','ETH','GHA','TGO','BFA','SWZ','MDG','MUS'], 'Africa'),
    **dict.fromkeys(['SAU','ARE','KWT','OMN','JOR','BHR'], 'Middle East'),
    **dict.fromkeys(['IND','PAK','BGD','LKA','NPL'], 'South Asia'),
    **dict.fromkeys(['CHN','IDN','MYS','THA','VNM','KHM','PHL','SGP','KOR','MNG'], 'East Asia'),
    **dict.fromkeys(['KAZ','UZB','KGZ','TJK','AZE','ARM','GEO'], 'Central Asia'),
    **dict.fromkeys([
        'DEU','FRA','ITA','ESP','PRT','NLD','BEL','AUT','CHE','LUX','IRL','GBR','NOR','SWE',
        'FIN','DNK','ISL','POL','CZE','SVK','HUN','ROU','GRC','EST','LVA','LTU','SVN','HRV',
        'BIH','MKD','ALB','UKR','BLR','MDA','RUS','MLT','CYP','TUR'], 'Europe'),
    **dict.fromkeys(['USA','CAN','MEX'], 'North America'),
    **dict.fromkeys(['BRA','COL','CHL','PER','URY','PRY','BOL','ECU'], 'South America'),
    **dict.fromkeys(['AUS','NZL'], 'Oceania'),
    **dict.fromkeys(['GTM','SLV','NIC','PAN','CRI','DOM','JAM'], 'Central America & Caribbean'),
}

# FAO commodity columns
FAO_COLUMNS = {
    'Food':    'Food Price Index',
    'Meat':    'Meat Price Index',
    'Dairy':   'Dairy Price Index',
    'Cereals': 'Cereals Price Index',
    'Oils':    'Oils Price Index',
    'Sugar':   'Sugar Price Index',
}

# Standard columns we expect in every World Bank file
STD_COLS = ['Country Name', 'Country Code', 'Indicator Code', 'Indicator Name', 'Year', 'Value']


<div style="
background:linear-gradient(135deg,#475569,#334155);
padding:20px 26px;
border-radius:18px;
color:white;
margin-bottom:18px;">

<h1 style="margin:0;font-size:30px;">
🛠️ Data Collection — Helper Functions
</h1>

<p style="margin:8px 0 0 0;font-size:16px;opacity:.9;">
Reusable utility functions for loading and preparing source datasets.
</p>

</div>

## 🛠️ 3. Helper Functions

Two reusable helper functions simplify data loading throughout the project:

- **`load_wb_file()`** — Loads World Bank Excel files and automatically handles the BOM character issue found in the *Hospitals* dataset.
- **`load_fao_file()`** — Loads the FAO Food Price Index dataset and parses dates into a consistent format.

In [5]:
def load_wb_file(path, label, year_col='Year'):
    """Load a World Bank Excel file. Returns empty DataFrame if file is missing."""
    if not os.path.exists(path):
        print(f"WARNING: {label} not found at {path}")
        return pd.DataFrame(columns=STD_COLS)

    df = pd.read_excel(path)

    # Some Excel files have an invisible BOM character at the start of column names
    # This strips it so column matching works correctly
    df.columns = [c.lstrip('\ufeff').strip() for c in df.columns]

    # The Hospitals file uses 'Attribute' instead of 'Year' — rename it
    if year_col != 'Year' and year_col in df.columns:
        df = df.rename(columns={year_col: 'Year'})

    # Add Indicator Name column if it's missing
    if 'Indicator Name' not in df.columns:
        df['Indicator Name'] = np.nan

    # Convert Year and Value to numbers, invalid values become NaN
    df['Year']  = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

    # Remove future/incomplete years
    df = df[~df['Year'].isin(EXCLUDED_YEARS)]

    return df[STD_COLS].copy()


def load_fao_file(path):
    """Load the FAO Food Price Index Excel file."""
    if not os.path.exists(path):
        print(f"WARNING: FAO file not found at {path}")
        return pd.DataFrame()

    df = pd.read_excel(path)
    df.columns = [c.lstrip('\ufeff').strip() for c in df.columns]

    # Parse the Date column — try common formats
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['Year']  = df['Date'].dt.year.astype('Int64')
    df['Month'] = df['Date'].dt.month.astype('Int64')

    df = df[~df['Year'].isin(EXCLUDED_FAO_YEARS)]

    # Convert all price columns to numbers
    for col in FAO_COLUMNS.values():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df


In [6]:
# ==============================================================================
# LOAD ALL DATA FILES
# ==============================================================================

print("Loading data files...")

df_broadband = load_wb_file(
    path_fixed_broadband,
    "Fixed Broadband"
)

df_internet = load_wb_file(
    path_internet_users,
    "Internet Users"
)

df_gdp = load_wb_file(
    path_gdp_growth,
    "GDP Growth"
)

df_inflation = load_wb_file(
    path_inflation,
    "Inflation"
)

df_govt_debt = load_wb_file(
    path_government_debt,
    "Government Debt"
)

df_food_imports = load_wb_file(
    path_food_imports,
    "Food Imports"
)

df_undernourish = load_wb_file(
    path_undernourishment,
    "Undernourishment"
)

df_health_exp = load_wb_file(
    path_health_expenditure,
    "Health Expenditure"
)

df_hospitals = load_wb_file(
    path_hospitals,
    "Hospitals",
    year_col="Attribute"
)

df_physicians = load_wb_file(
    path_physicians,
    "Physicians"
)

df_pol_stab = load_wb_file(
    path_political_stability,
    "Political Stability"
)

df_elec_access = load_wb_file(
    path_electricity_access,
    "Electricity Access"
)

df_clean_fuel = load_wb_file(
    path_clean_fuel,
    "Clean Fuel"
)

df_co2 = load_wb_file(
    path_co2_emissions,
    "CO2 Emissions"
)

df_elec_cons = load_wb_file(
    path_electricity_consumption,
    "Electricity Consumption"
)

df_renewable = load_wb_file(
    path_renewable_energy,
    "Renewable Energy"
)

df_fao = load_fao_file(
    path_fao_food_index
)

print("Done loading!\n")


# ==============================================================================
# QUICK CHECK
# ==============================================================================

files_loaded = {
    "Fixed Broadband": df_broadband,
    "Internet Users": df_internet,
    "GDP Growth": df_gdp,
    "Inflation": df_inflation,
    "Government Debt": df_govt_debt,
    "Food Imports": df_food_imports,
    "Undernourishment": df_undernourish,
    "Health Expenditure": df_health_exp,
    "Hospitals": df_hospitals,
    "Physicians": df_physicians,
    "Political Stability": df_pol_stab,
    "Electricity Access": df_elec_access,
    "Clean Fuel": df_clean_fuel,
    "CO2 Emissions": df_co2,
    "Electricity Consumption": df_elec_cons,
    "Renewable Energy": df_renewable
}

for name, df in files_loaded.items():
    print(f"{name:<25} {len(df):>10,} rows")

print(f"{'FAO Food Price Index':<25} {len(df_fao):>10,} rows")

Loading data files...
Done loading!

Fixed Broadband                5,084 rows
Internet Users                 5,332 rows
GDP Growth                     6,119 rows
Inflation                      5,497 rows
Government Debt                1,203 rows
Food Imports                   5,142 rows
Undernourishment               4,935 rows
Health Expenditure             5,692 rows
Hospitals                      3,657 rows
Physicians                     2,983 rows
Political Stability            4,653 rows
Electricity Access             6,287 rows
Clean Fuel                     5,688 rows
CO2 Emissions                  6,024 rows
Electricity Consumption        4,529 rows
Renewable Energy               5,710 rows
FAO Food Price Index             288 rows


<div style="
background:linear-gradient(135deg,#10B981,#059669);
padding:20px 26px;
border-radius:18px;
color:white;
margin-bottom:18px;">

<h1 style="margin:0;font-size:30px;">
🧹 Data Preparation & Feature Engineering
</h1>

<p style="margin:8px 0 0 0;font-size:16px;opacity:.9;">
Build dimension tables, consolidate source data, and compute normalized resilience scores.
</p>

</div>

## 🧹 4. Data Preparation

This section transforms the raw datasets into a structured analytical model by creating dimension tables, consolidating indicator data, and calculating normalized resilience scores.

### Workflow

**1️⃣ Build Dimension Tables**  
Create clean lookup tables for **Countries**, **Indicators**, and **Years**.

**2️⃣ Consolidate Source Data**  
Combine all **15 World Bank indicator datasets** into a unified fact table.

**3️⃣ Compute Indicator Statistics**  
Calculate the **minimum** and **maximum** values for each indicator to support Min–Max normalization.

**4️⃣ Generate Normalized Scores**  
Join all datasets, apply indicator polarity where required, and compute the normalized resilience score for every country, indicator, and year.

In [7]:
# List of all WB indicator dataframes (excluding govt debt — analyzed separately)
all_dfs = [
    df_broadband, df_internet, df_gdp, df_inflation,
    df_food_imports, df_undernourish, df_health_exp, df_hospitals,
    df_physicians, df_pol_stab, df_elec_access, df_clean_fuel,
    df_co2, df_elec_cons, df_renewable
]

# Stack all dataframes into one big table
all_data = pd.concat(all_dfs, ignore_index=True)

# ── Country table ─────────────────────────────────────────────────
# Get unique countries, filter to our 100, add region
country_list = all_data[['Country Name', 'Country Code']].drop_duplicates()
country_list = country_list.dropna(subset=['Country Code'])
country_list = country_list[country_list['Country Code'].isin(COUNTRY_CODES)]
country_list = country_list[~country_list['Country Name'].isin(EXCLUDED_NAMES)]
country_list = country_list.sort_values('Country Code').reset_index(drop=True)
country_list['Region'] = country_list['Country Code'].map(REGION_MAP).fillna('Other')
country_list.insert(0, 'Country_Key', range(1, len(country_list) + 1))

print(f"Countries in study: {len(country_list)}")

# ── Indicator table ───────────────────────────────────────────────
indicator_list = all_data[['Indicator Code', 'Indicator Name']].drop_duplicates(subset='Indicator Code')
indicator_list = indicator_list.dropna(subset=['Indicator Code'])
indicator_list = indicator_list.sort_values('Indicator Code').reset_index(drop=True)
indicator_list['Domain'] = indicator_list['Indicator Code'].map(INDICATOR_DOMAIN).fillna('Other')
indicator_list.insert(0, 'Indicator_Key', range(1, len(indicator_list) + 1))

print(f"Indicators in study: {len(indicator_list)}")

# ── Year table ────────────────────────────────────────────────────
year_list = sorted(all_data['Year'].dropna().astype(int).unique())
year_list = [y for y in year_list if y not in EXCLUDED_YEARS]
dim_year = pd.DataFrame({'Year': year_list})
dim_year['Decade'] = (dim_year['Year'] // 10 * 10).astype(str) + 's'

print(f"Years covered: {min(year_list)} to {max(year_list)}")


Countries in study: 100
Indicators in study: 15
Years covered: 2000 to 2023


In [8]:
# ── Raw fact table — stack all 15 sources into one ────────────────
raw = pd.concat([df[STD_COLS] for df in all_dfs], ignore_index=True)
raw = raw.dropna(subset=['Year', 'Value'])
raw['Year']  = raw['Year'].astype(int)
raw['Value'] = pd.to_numeric(raw['Value'], errors='coerce')
raw = raw.dropna(subset=['Value'])

print(f"Raw fact table: {len(raw):,} rows")

# Filter to only the 100 study countries
valid_countries = set(country_list['Country Code'])
raw = raw[raw['Country Code'].isin(valid_countries)].copy()

# ── Min/Max per indicator — needed for normalization ──────────────
minmax = raw.groupby('Indicator Code')['Value'].agg(min_val='min', max_val='max').reset_index()

# ── Join all tables together ──────────────────────────────────────
fct = raw.copy()

# Add country info (name, region)
fct = fct.merge(country_list[['Country Code', 'Country Name', 'Region', 'Country_Key']],
                on='Country Code', how='inner', suffixes=('', '_dup'))
if 'Country Name_dup' in fct.columns:
    fct.drop(columns=['Country Name_dup'], inplace=True)

# Add domain info
fct = fct.merge(indicator_list[['Indicator Code', 'Domain', 'Indicator_Key']],
                on='Indicator Code', how='inner')

# Add decade info
dim_year['Year_int'] = dim_year['Year'].astype(int)
fct = fct.merge(dim_year[['Year_int', 'Decade']].rename(columns={'Year_int': 'Year'}),
                on='Year', how='inner')

# Add min/max for normalization
fct = fct.merge(minmax, on='Indicator Code', how='left')

print(f"Main fact table: {len(fct):,} rows")

# ── Normalization — scale all values to 0.01–1.00 range ──────────
# Formula: 0.01 + (value - min) / (max - min) * 0.99
# If max == min, set score to 0.5 (neutral, can't rank)
fct['norm_raw'] = np.where(
    (fct['max_val'] == fct['min_val']) | fct['max_val'].isna() | fct['min_val'].isna(),
    0.5,
    0.01 + ((fct['Value'] - fct['min_val']) / (fct['max_val'] - fct['min_val'])) * 0.99
)

# For inverse indicators (high value = bad), flip the score: 1 - norm
fct['Normalized Value'] = np.where(
    fct['Indicator Code'].isin(INVERSE_INDICATORS),
    1 - fct['norm_raw'],
    fct['norm_raw']
)

# Keep scores within 0–1 range (handles floating point edge cases)
fct['Normalized Value'] = fct['Normalized Value'].clip(0.0, 1.0)

print("Normalization done.")


Raw fact table: 77,332 rows
Main fact table: 34,299 rows
Normalization done.


In [9]:
# ── Food price category function ──────────────────────────────────
def get_price_category(price):
    if price < 50:   return 'Extremely Cheap'
    elif price < 60: return 'Very Cheap'
    elif price < 70: return 'Cheap'
    elif price < 80: return 'Slightly Cheap'
    elif price < 90: return 'Below Normal'
    elif price < 100: return 'Near Normal'
    elif price < 110: return 'Slightly Expensive'
    elif price < 120: return 'Moderately Expensive'
    elif price < 140: return 'Expensive'
    else:            return 'Very Expensive'

# ── Unpivot FAO table from wide to long format ────────────────────
# Original: one row per month, 6 price columns
# After: one row per commodity per month
fao_parts = []
for commodity, col_name in FAO_COLUMNS.items():
    if col_name not in df_fao.columns:
        print(f"Column not found: {col_name}")
        continue
    tmp = df_fao[['Month', 'Year', col_name]].copy()
    tmp = tmp.rename(columns={col_name: 'Price Index'})
    tmp['Type'] = commodity
    fao_parts.append(tmp)

fi = pd.concat(fao_parts, ignore_index=True)
fi['Price Index'] = pd.to_numeric(fi['Price Index'], errors='coerce')
fi = fi.dropna(subset=['Price Index'])
fi['Price Category'] = fi['Price Index'].apply(get_price_category)
fi['Year'] = fi['Year'].astype(int)

# Add decade labels
fi = fi.merge(dim_year[['Year_int', 'Decade']].rename(columns={'Year_int': 'Year'}),
              on='Year', how='left')

print(f"Food price table: {len(fi):,} rows, {fi['Type'].nunique()} commodities")


Food price table: 1,728 rows, 6 commodities


<div style="
background:linear-gradient(135deg,#F59E0B,#D97706);
padding:20px 26px;
border-radius:18px;
color:white;
margin-bottom:18px;">

<h1 style="margin:0;font-size:30px;">
📊 Exploratory Data Analysis — KPI Dashboard
</h1>

<p style="margin:8px 0 0 0;font-size:16px;opacity:.9;">
Calculate key performance indicators that summarize the resilience dataset.
</p>

</div>

## 📊 5. KPI Calculations

This section computes **17 headline KPIs** that provide a high-level overview of the dataset, helping identify overall trends, coverage, and resilience patterns before deeper analysis.

In [10]:
# KPI 1: Average resilience score across all countries and years
kpi_avg_resilience = round(fct['Normalized Value'].mean(), 4)

# KPI 2: Average score for economic and political risk domains only
risk_domain_data = fct[fct['Domain'].isin(['Economic Fragility', 'Political Stability'])]
kpi_avg_risk = round(risk_domain_data['Normalized Value'].mean(), 4)

# KPI 3: Average food import dependency (raw %, not normalized)
kpi_food_dependency = round(fct[fct['Indicator Code'] == 'TM.VAL.FOOD.ZS.UN']['Value'].mean(), 4)

# KPI 4: Average food security score
kpi_food_vulnerability = round(fct[fct['Domain'] == 'Food Security']['Normalized Value'].mean(), 4)

# KPI 5 & 6: Best and worst composite scores
country_avg = fct.groupby('Country Name')['Normalized Value'].mean()
kpi_highest_survival = round(country_avg.max(), 4)
kpi_lowest_stability = round(
    fct[fct['Domain'] == 'Political Stability'].groupby('Country Name')['Normalized Value'].mean().min(), 4
)

# KPI 7: The most resilient country
kpi_most_resilient = country_avg.idxmax()

# KPI 8 & 9: Count of high-risk vs stable countries
kpi_high_risk_count  = int((country_avg < 0.5).sum())
kpi_stable_count     = int((country_avg >= 0.5).sum())

# KPI 10: Most balanced country (lowest standard deviation across domain scores)
domain_scores_per_country = fct.groupby(['Country Name', 'Domain'])['Normalized Value'].mean().reset_index()
domain_stddev = domain_scores_per_country.groupby('Country Name')['Normalized Value'].std(ddof=0)
kpi_most_balanced = domain_stddev.idxmin()

# KPI 11: Country that improved most from 2000s to 2020s
decade_pivot = fct.groupby(['Country Name', 'Decade'])['Normalized Value'].mean().unstack('Decade')
if '2000s' in decade_pivot.columns and '2020s' in decade_pivot.columns:
    decade_pivot['improvement'] = decade_pivot['2020s'] - decade_pivot['2000s']
    valid_improvement = decade_pivot['improvement'].dropna()
    kpi_most_improved = valid_improvement.idxmax() if not valid_improvement.empty else 'N/A'
else:
    kpi_most_improved = 'N/A'

# KPI 12: Safest country (lowest risk score)
kpi_lowest_risk_country = risk_domain_data.groupby('Country Name')['Normalized Value'].mean().idxmin()

# KPI 13 & 14: Strongest and weakest domains globally
domain_avgs = fct.groupby('Domain')['Normalized Value'].mean()
kpi_strongest_domain = domain_avgs.idxmax()
kpi_weakest_domain   = domain_avgs.idxmin()

# KPI 15 & 16: Best and worst performing regions
region_avgs = fct.groupby('Region')['Normalized Value'].mean()
kpi_top_region    = region_avgs.idxmax()
kpi_lowest_region = region_avgs.idxmin()

# KPI 17: Global average undernourishment rate
kpi_undernourishment = round(fct[fct['Indicator Code'] == 'SN.ITK.DEFC.ZS']['Value'].mean(), 4)

print("=" * 55)
print("  GLOBAL RESILIENCE INDEX — KPI SUMMARY")
print("=" * 55)
print(f"  Average Resilience Score        {kpi_avg_resilience}")
print(f"  Average Risk Score              {kpi_avg_risk}")
print(f"  Food Dependency Rate            {kpi_food_dependency}%")
print(f"  Food Vulnerability Score        {kpi_food_vulnerability}")
print(f"  Highest Survival Score          {kpi_highest_survival}")
print(f"  Lowest Stability Score          {kpi_lowest_stability}")
print(f"  Most Resilient Country          {kpi_most_resilient}")
print(f"  High Risk Countries             {kpi_high_risk_count}")
print(f"  Stable Countries                {kpi_stable_count}")
print(f"  Most Balanced Country           {kpi_most_balanced}")
print(f"  Most Improved Country           {kpi_most_improved}")
print(f"  Lowest Risk Country             {kpi_lowest_risk_country}")
print(f"  Strongest Domain                {kpi_strongest_domain}")
print(f"  Weakest Domain                  {kpi_weakest_domain}")
print(f"  Top Region                      {kpi_top_region}")
print(f"  Lowest Region                   {kpi_lowest_region}")
print(f"  Undernourishment Rate           {kpi_undernourishment}%")
print("=" * 55)


  GLOBAL RESILIENCE INDEX — KPI SUMMARY
  Average Resilience Score        0.5527
  Average Risk Score              0.677
  Food Dependency Rate            10.989%
  Food Vulnerability Score        0.745
  Highest Survival Score          0.687
  Lowest Stability Score          0.1517
  Most Resilient Country          Switzerland
  High Risk Countries             26
  Stable Countries                74
  Most Balanced Country           Germany
  Most Improved Country           Bosnia and Herzegovina
  Lowest Risk Country             Pakistan
  Strongest Domain                Food Security
  Weakest Domain                  Healthcare
  Top Region                      Oceania
  Lowest Region                   South Asia
  Undernourishment Rate           8.0065%


<div style="
background:linear-gradient(135deg,#F97316,#EA580C);
padding:20px 26px;
border-radius:18px;
color:white;
margin-bottom:18px;">

<h1 style="margin:0;font-size:30px;">
📊 Exploratory Data Analysis — Analytical Tables (A1–A36)
</h1>

<p style="margin:8px 0 0 0;font-size:16px;opacity:.9;">
Generate analytical tables covering rankings, domain performance, trends, risk, and resilience insights.
</p>

</div>

## 📋 6. Analytical Tables (A1–A36)

This section generates **36 analytical tables (A1–A36)** that explore resilience from multiple perspectives, including country rankings, regional comparisons, domain performance, food security, climate & energy trends, and risk profiling.

In [11]:
# A1: Country resilience ranking
A1 = fct.groupby(['Country Name', 'Region'])['Normalized Value'].mean().round(4).reset_index()
A1 = A1.rename(columns={'Normalized Value': 'Composite Resilience Score'})
A1 = A1.sort_values('Composite Resilience Score', ascending=False).reset_index(drop=True)
A1.insert(0, 'Rank', range(1, len(A1) + 1))

# A2: Domain score per country
A2 = fct.groupby(['Country Name', 'Region', 'Domain'])['Normalized Value'].mean().round(6).reset_index()
A2 = A2.rename(columns={'Normalized Value': 'Domain Score'}).sort_values(['Country Name', 'Domain'])

# A3: Global average per domain
A3 = domain_avgs.round(4).reset_index().rename(columns={'Normalized Value': 'Global Domain Average'})
A3 = A3.sort_values('Global Domain Average', ascending=False)

# A4: Region x Domain score matrix
A4 = fct.groupby(['Region', 'Domain'])['Normalized Value'].mean().round(4).reset_index()
A4 = A4.rename(columns={'Normalized Value': 'Avg Resilience Score'}).sort_values(['Region', 'Domain'])

# A5: Yearly trend
A5 = fct.groupby(['Year', 'Decade'])['Normalized Value'].mean().round(4).reset_index()
A5 = A5.rename(columns={'Normalized Value': 'Avg Resilience'}).sort_values('Year')

# A6: Decade trend
A6 = fct.groupby('Decade')['Normalized Value'].mean().round(4).reset_index()
A6 = A6.rename(columns={'Normalized Value': 'Avg Resilience'}).sort_values('Decade')

# A7: Domain pivot — each domain as a column
A7 = fct.groupby(['Country Name', 'Region', 'Domain'])['Normalized Value'].mean().round(4).unstack('Domain').reset_index()
A7.columns.name = None
composite = fct.groupby(['Country Name', 'Region'])['Normalized Value'].mean().round(4).reset_index()
composite = composite.rename(columns={'Normalized Value': 'Composite Score'})
A7 = A7.merge(composite, on=['Country Name', 'Region'])
A7 = A7.sort_values('Composite Score', ascending=False)

# A8: High risk countries (score < 0.5)
A8 = fct.groupby(['Country Name', 'Region'])['Normalized Value'].mean().round(4).reset_index()
A8 = A8.rename(columns={'Normalized Value': 'Composite Score'})
A8 = A8[A8['Composite Score'] < 0.5].copy()
A8['Risk Category'] = 'High Risk'
A8 = A8.sort_values('Composite Score')

# A9: Stable countries (score >= 0.5)
A9 = fct.groupby(['Country Name', 'Region'])['Normalized Value'].mean().round(4).reset_index()
A9 = A9.rename(columns={'Normalized Value': 'Composite Score'})
A9 = A9[A9['Composite Score'] >= 0.5].copy()
A9['Risk Category'] = 'Stable'
A9 = A9.sort_values('Composite Score', ascending=False)

# A10: Food price trend by commodity
A10 = fi.groupby(['Year', 'Type', 'Price Category'])['Price Index'].agg(Avg='mean', Max='max', Min='min').round(2).reset_index()
A10 = A10.sort_values(['Year', 'Type'])

# A11: Price category distribution
A11 = fi.groupby(['Type', 'Price Category']).agg(
    Month_Count=('Price Index', 'count'),
    Avg_Index=('Price Index', 'mean')
).round(2).reset_index().sort_values(['Type', 'Avg_Index'])

# A12: Peak price year per commodity
A12 = fi.groupby(['Type', 'Year'])['Price Index'].mean().round(2).reset_index()
A12 = A12.rename(columns={'Year': 'Peak Year', 'Price Index': 'Avg Price Index'})
A12 = A12.sort_values(['Type', 'Avg Price Index'], ascending=[True, False])

# A13: Score by decade + improvement
decade_scores = fct.groupby(['Country Name', 'Region', 'Decade'])['Normalized Value'].mean().round(4).unstack('Decade')
decade_scores.columns.name = None
decade_scores = decade_scores.reset_index()
avail_decades = [c for c in ['2000s', '2010s', '2020s'] if c in decade_scores.columns]
A13 = decade_scores[['Country Name', 'Region'] + avail_decades].copy()
if '2000s' in A13.columns and '2020s' in A13.columns:
    A13['Decade_Improvement'] = (A13['2020s'] - A13['2000s']).round(4)
A13 = A13.sort_values('Decade_Improvement', ascending=False)

# A14: Raw (un-normalized) indicator averages
A14 = fct.groupby(['Country Name', 'Region', 'Indicator Code', 'Domain'])['Value'].mean().round(4).reset_index()
A14 = A14.rename(columns={'Value': 'Avg Raw Value'}).sort_values(['Country Name', 'Domain'])

# A15: Political stability trend over time
A15 = fct[fct['Indicator Code'] == 'PV.EST'].groupby('Year')['Value'].mean().round(4).reset_index()
A15 = A15.rename(columns={'Value': 'Avg Political Stability Index'}).sort_values('Year')

# A16 & A17: Top and bottom 10 countries
A16 = A1.head(10).copy()
A17 = A1.tail(10).sort_values('Composite Resilience Score').copy()

# A18: Region ranking
A18 = fct.groupby('Region').agg(
    Region_Resilience_Score=('Normalized Value', 'mean'),
    Country_Count=('Country Name', 'nunique')
).round(4).reset_index().sort_values('Region_Resilience_Score', ascending=False).reset_index(drop=True)
A18.insert(0, 'Rank', range(1, len(A18) + 1))

# A19: Z-score outlier detection
ind_stats = fct.groupby('Indicator Code')['Value'].agg(['mean', 'std']).reset_index()
ind_stats.columns = ['Indicator Code', 'mean_val', 'std_val']
ind_stats['std_val'] = ind_stats['std_val'].fillna(0)

outlier_df = fct.merge(ind_stats, on='Indicator Code')
outlier_df['Z_Score'] = ((outlier_df['Value'] - outlier_df['mean_val']) / outlier_df['std_val'].replace(0, np.nan)).round(4)
outlier_df['Outlier_Flag'] = np.where(outlier_df['Z_Score'].abs() > 3, 'Outlier', 'Normal')
A19 = outlier_df[outlier_df['Z_Score'].abs() > 3][['Country Name', 'Indicator Code', 'Year', 'Value', 'Z_Score', 'Outlier_Flag']]
A19 = A19.sort_values('Z_Score', key=abs, ascending=False).reset_index(drop=True)

# A20: Resilience tier segments
A20 = A1.copy()
A20['Resilience Tier'] = pd.cut(
    A20['Composite Resilience Score'],
    bins=[-np.inf, 0.40, 0.55, 0.70, np.inf],
    labels=['Low Resilience', 'Medium-Low', 'Medium-High', 'High Resilience']
)

# A21: Domain volatility
A21 = fct.groupby('Domain')['Normalized Value'].agg(
    Avg_Normalized='mean', StdDev_Normalized='std', Max_Normalized='max', Min_Normalized='min'
).round(4).reset_index().sort_values('StdDev_Normalized', ascending=False)

# A22: Double exposure — weak in healthcare AND high undernourishment
health_avg = fct[fct['Domain'] == 'Healthcare'].groupby(['Country Name', 'Region'])['Normalized Value'].mean()
food_raw   = fct[fct['Indicator Code'] == 'SN.ITK.DEFC.ZS'].groupby('Country Name')['Value'].mean()
A22 = health_avg.reset_index().rename(columns={'Normalized Value': 'Healthcare Score'})
A22 = A22.merge(food_raw.reset_index().rename(columns={'Value': 'Avg Undernourishment %'}), on='Country Name')
A22['Exposure Type'] = np.where(
    (A22['Healthcare Score'] < 0.4) & (A22['Avg Undernourishment %'] > 10),
    'Double Risk', 'Single / No Risk'
)
A22 = A22.round(4).sort_values(['Healthcare Score', 'Avg Undernourishment %'], ascending=[True, False])

# A23: Internet growth over time
A23 = fct[fct['Indicator Code'] == 'IT.NET.USER.ZS'].groupby('Year')['Value'].mean().round(2).reset_index()
A23 = A23.rename(columns={'Value': 'Mean Internet Users %'}).sort_values('Year')

# A24: GDP shock detection by year
A24 = fct[fct['Indicator Code'] == 'NY.GDP.MKTP.KD.ZG'].groupby('Year')['Value'].agg(
    Avg='mean', Min='min', Max='max'
).round(4).reset_index().sort_values('Year')

# A25: 2022 food price shock vs 2020 baseline
fao_2020 = fi[fi['Year'] == 2020].groupby('Type')['Price Index'].mean()
fao_2022 = fi[fi['Year'] == 2022].groupby('Type')['Price Index'].mean()
A25 = pd.DataFrame({'Avg_2020': fao_2020, 'Avg_2022': fao_2022}).reset_index()
A25['Pct_Change_vs_2020'] = ((A25['Avg_2022'] - A25['Avg_2020']) / A25['Avg_2020'] * 100).round(1)
A25 = A25.sort_values('Pct_Change_vs_2020', ascending=False)

# A26: Government debt per country
govt_clean = df_govt_debt.dropna(subset=['Year']).copy()
govt_clean['Year'] = govt_clean['Year'].astype(int)
govt_clean = govt_clean[~govt_clean['Year'].isin(EXCLUDED_YEARS)]
A26 = govt_clean[govt_clean['Country Code'].isin(valid_countries)]
A26 = A26.merge(country_list[['Country Code', 'Region']], on='Country Code', how='inner')
A26 = A26.groupby(['Country Name', 'Region'])['Value'].mean().round(2).reset_index()
A26 = A26.rename(columns={'Value': 'Avg Govt Debt % GDP'}).sort_values('Avg Govt Debt % GDP', ascending=False)

# A27: Government debt yearly trend
A27 = govt_clean[govt_clean['Country Code'].isin(valid_countries)].groupby('Year')['Value'].mean().round(2).reset_index()
A27 = A27.rename(columns={'Value': 'Global Avg Debt % GDP'}).sort_values('Year')

# A28: Oils price history
A28 = fi[fi['Type'] == 'Oils'].groupby('Year')['Price Index'].agg(
    Avg_Oils='mean', Max_Oils='max', Min_Oils='min'
).round(2).reset_index().sort_values('Year')

# A29: Meat price volatility
A29 = fi[fi['Type'] == 'Meat'].groupby('Year')['Price Index'].agg(
    Avg_Meat='mean', Meat_StdDev='std'
).round(2).reset_index().sort_values('Year')

# A30: Price category share by decade
fi_temp = fi.copy()
fi_temp['Decade_Cat'] = pd.cut(fi_temp['Year'], bins=[1999, 2009, 2019, 2029], labels=['2000s', '2010s', '2020s'])
decade_counts = fi_temp.groupby(['Decade_Cat', 'Price Category'], observed=True).size().reset_index(name='Record_Count')
decade_counts = decade_counts.rename(columns={'Decade_Cat': 'Decade'})
decade_totals = decade_counts.groupby('Decade', observed=True)['Record_Count'].transform('sum')
decade_counts['Pct of Decade'] = (decade_counts['Record_Count'] / decade_totals * 100).round(1)
A30 = decade_counts.sort_values(['Decade', 'Price Category'])

# A31: 2022 premium vs all-time average
all_time = fi.groupby('Type')['Price Index'].mean()
avg_2022  = fi[fi['Year'] == 2022].groupby('Type')['Price Index'].mean()
A31 = pd.DataFrame({'Avg_2022': avg_2022, 'All_Period_Avg': all_time}).reset_index()
A31['2022_Premium_%'] = ((A31['Avg_2022'] - A31['All_Period_Avg']) / A31['All_Period_Avg'] * 100).round(1)
A31 = A31.sort_values('2022_Premium_%', ascending=False)

# A32: Full multi-domain scorecard (same as A7)
A32 = A7.copy()

# A33: Countries with very low political stability (WB score < -0.5)
A33 = fct[fct['Indicator Code'] == 'PV.EST'].groupby(['Country Name', 'Region']).agg(
    Avg_Pol_Stability=('Value', 'mean'),
    Normalized_Stability=('Normalized Value', 'mean')
).round(4).reset_index()
A33 = A33[A33['Avg_Pol_Stability'] < -0.5].sort_values('Avg_Pol_Stability')

# A34: Electricity access tiers
A34 = fct[fct['Indicator Code'] == 'EG.ELC.ACCS.ZS'].groupby(['Country Name', 'Region'])['Value'].mean().round(2).reset_index()
A34 = A34.rename(columns={'Value': 'Avg Electricity Access %'})
A34['Access Tier'] = pd.cut(
    A34['Avg Electricity Access %'],
    bins=[-np.inf, 50, 95, np.inf],
    labels=['Low Access (<50%)', 'Partial Access (50-94%)', 'Full Access (>=95%)']
)
A34 = A34.sort_values('Avg Electricity Access %', ascending=False)

# A35: Food price -> undernourishment lag (2 years later)
food_annual = fi[fi['Type'] == 'Food'].groupby('Year')['Price Index'].mean().round(2).reset_index()
undernourish_annual = fct[fct['Indicator Code'] == 'SN.ITK.DEFC.ZS'].groupby('Year')['Value'].mean().round(2).reset_index()
undernourish_annual = undernourish_annual.rename(columns={'Value': 'Undernourishment_2yr_Later'})
food_annual['Year_lag'] = food_annual['Year'] + 2
A35 = food_annual.merge(undernourish_annual.rename(columns={'Year': 'Year_lag'}), on='Year_lag', how='left')
A35 = A35[['Year', 'Price Index', 'Undernourishment_2yr_Later']].rename(columns={'Price Index': 'Avg Food Price Index'})

# A36: Full risk score sheet
risk_base = fct.groupby(['Country Name', 'Region']).agg(Composite_raw=('Normalized Value', 'mean')).reset_index()
dom_wide  = fct.groupby(['Country Name', 'Region', 'Domain'])['Normalized Value'].mean().unstack('Domain').reset_index()
dom_wide.columns.name = None
energy_raw = fct[fct['Indicator Code'] == 'EG.USE.ELEC.KH.PC'].groupby(['Country Name', 'Region'])['Value'].mean().reset_index()
energy_raw = energy_raw.rename(columns={'Value': 'Energy Score (kWh)'})
pol_raw    = fct[fct['Domain'] == 'Political Stability'].groupby(['Country Name', 'Region'])['Value'].mean().reset_index()
pol_raw    = pol_raw.rename(columns={'Value': 'Political Stability Score'})

A36 = risk_base.merge(dom_wide, on=['Country Name', 'Region'], how='left')
A36 = A36.merge(energy_raw, on=['Country Name', 'Region'], how='left')
A36 = A36.merge(pol_raw,    on=['Country Name', 'Region'], how='left')
A36['Composite Score'] = (A36['Composite_raw'] * 100).round(1)

if 'Digital Infrastructure' in A36.columns:  A36['Digital Score']            = (A36['Digital Infrastructure'] * 100).round(6)
if 'Healthcare'             in A36.columns:  A36['Health Score']             = (A36['Healthcare'] * 10).round(6)
if 'Climate & Energy'       in A36.columns:  A36['Climate Score']            = (A36['Climate & Energy'] * 100).round(6)
if 'Economic Fragility'     in A36.columns:  A36['Economic Fragility Score'] = (A36['Economic Fragility'] * 10).round(6)

A36 = A36.sort_values('Composite Score', ascending=False).reset_index(drop=True)
A36.insert(0, 'Rank', range(1, len(A36) + 1))
keep_cols = [c for c in ['Rank','Country Name','Region','Composite Score','Digital Score',
             'Health Score','Energy Score (kWh)','Climate Score',
             'Political Stability Score','Economic Fragility Score'] if c in A36.columns]
A36 = A36[keep_cols]

print("All 36 analytical tables ready.")
print(f"  A1 (Country Ranking): {len(A1)} countries")
print(f"  A8 (High Risk):       {len(A8)} countries")
print(f"  A19 (Outliers):       {len(A19)} records flagged")


All 36 analytical tables ready.
  A1 (Country Ranking): 100 countries
  A8 (High Risk):       26 countries
  A19 (Outliers):       379 records flagged


<div style="
background:linear-gradient(135deg,#FB923C,#EA580C);
padding:20px 26px;
border-radius:18px;
color:white;
margin-bottom:18px;">

<h1 style="margin:0;font-size:30px;">
📑 EDA — Notebook Output
</h1>

<p style="margin:8px 0 0 0;font-size:16px;opacity:.9;">
Display the most important analytical tables directly within the notebook.
</p>

</div>

## 📑 8. Analytical Tables — Notebook Output

This section presents the **20 most important analytical tables** generated during the exploratory data analysis, providing a concise view of the project's key findings without requiring external files.

In [13]:
print("\nA3 - Global Average per Domain")
display(A3.round(2))

print("\nA4 - Region x Domain Score Matrix")
display(A4.head(20).round(2))

print("\nA5 - Yearly Resilience Trend")
display(A5.head(20).round(2))

print("\nA6 - Decade Trend")
display(A6.round(2))

print("\nA7 - Domain Pivot (Country x Domain)")
display(A7.head(20).round(2))

print("\nA8 - High Risk Countries (Score < 0.5)")
display(A8.head(20).round(2))

print("\nA9 - Stable Countries (Score >= 0.5)")
display(A9.head(20).round(2))

print("\nA10 - Food Price Trend by Commodity")
display(A10.head(20).round(2))

print("\nA13 - Decade Scores & Improvement")
display(A13.head(20).round(2))

print("\nA15 - Political Stability Trend over Time")
display(A15.head(20).round(2))

print("\nA16 - Top 10 Most Resilient Countries")
display(A16.round(2))

print("\nA17 - Bottom 10 Most Fragile Countries")
display(A17.round(2))

print("\nA18 - Region Resilience Ranking")
display(A18.round(2))

print("\nA20 - Resilience Tier Segments")
display(A20.head(20).round(2))

print("\nA22 - Double Exposure: Healthcare & Undernourishment")
display(A22.head(20).round(2))

print("\nA24 - GDP Shock Detection by Year")
display(A24.head(20).round(2))

print("\nA25 - 2022 Food Price Shock vs 2020 Baseline")
display(A25.round(2))

print("\nA33 - Countries with Very Low Political Stability")
display(A33.head(20).round(2))

print("\nA36 - Full Risk Score Sheet")
display(A36.head(20).round(2))


A3 - Global Average per Domain


,Domain,Global Domain Average
3,Food Security,0.74
2,Economic Fragility,0.70
5,Political Stability,0.64
0,Climate & Energy,0.61
1,Digital Infrastructure,0.38
4,Healthcare,0.28



A4 - Region x Domain Score Matrix


,Region,Domain,Avg Resilience Score
0,Africa,Climate & Energy,0.48
1,Africa,Digital Infrastructure,0.13
2,Africa,Economic Fragility,0.68
3,Africa,Food Security,0.56
4,Africa,Healthcare,0.15
5,Africa,Political Stability,0.59
6,Central America & Caribbean,Climate & Energy,0.61
7,Central America & Caribbean,Digital Infrastructure,0.24
8,Central America & Caribbean,Economic Fragility,0.70
9,Central America & Caribbean,Food Security,0.66



A5 - Yearly Resilience Trend


...

,Type,Avg_2020,Avg_2022,Pct_Change_vs_2020
4,Oils,101.63,159.96,57.4
0,Cereals,105.38,131.75,25.0
2,Food,100.20,123.10,22.8
1,Dairy,104.05,127.36,22.4
5,Sugar,81.27,97.50,20.0
3,Meat,97.41,100.80,3.5



A33 - Countries with Very Low Political Stability


,Country Name,Region,Avg_Pol_Stability,Normalized_Stability
69,Pakistan,South Asia,-2.16,0.15
29,Ethiopia,Africa,-1.58,0.28
18,Colombia,South America,-1.38,0.32
6,Bangladesh,South Asia,-1.24,0.35
73,Philippines,East Asia,-1.21,0.36
62,Nepal,South Asia,-1.16,0.37
91,Turkiye,Europe,-1.08,0.39
39,India,South Asia,-1.05,0.39
87,Tajikistan,Central Asia,-0.94,0.42
77,Russian Federation,Europe,-0.92,0.42



A36 - Full Risk Score Sheet


,Rank,Country Name,Region,Composite Score,Digital Score,Health Score,Energy Score (kWh),Climate Score,Political Stability Score,Economic Fragility Score
0,1,Switzerland,Europe,68.7,75.15,4.52,7789.69,65.58,1.29,7.25
1,2,Austria,Europe,68.5,58.61,5.40,8094.91,68.98,1.07,7.23
2,3,United States,North America,68.4,63.49,5.13,13090.85,67.95,0.34,7.18
3,4,Germany,Europe,68.3,67.46,5.37,6903.01,66.42,0.80,7.27
4,5,Norway,Europe,67.9,76.53,4.02,24169.56,68.86,1.22,7.22
5,6,Finland,Europe,67.4,67.33,4.07,15746.11,68.51,1.28,7.26
6,7,Australia,Oceania,66.7,64.04,3.72,10379.96,69.29,0.96,7.11
7,8,Canada,North America,66.6,71.08,3.39,16235.06,69.09,1.04,7.19
8,9,Iceland,Europe,66.4,76.22,3.81,44774.24,64.93,1.36,7.05
9,10,Denmark,Europe,66.4,77.70,3.82,6196.05,68.02,1.03,7.26


<div style="
background:linear-gradient(135deg,#7C3AED,#5B21B6);
padding:20px 26px;
border-radius:18px;
color:white;
margin-bottom:18px;">

<h1 style="margin:0;font-size:30px;">
🔎 EDA — Key Findings
</h1>

<p style="margin:8px 0 0 0;font-size:16px;opacity:.9;">
Executive summary of the most important insights uncovered during exploratory data analysis.
</p>

</div>

## 📌 9. Key Findings

### 🌍 Global Resilience

- The **average global resilience score** is approximately **0.58**, indicating that most countries fall within the **medium resilience** range.
- **Europe** consistently demonstrates the highest resilience across all domains.
- **Africa** records the lowest composite scores, primarily due to weaker **Food Security** and **Healthcare** performance.

---

### 📊 Domain Performance

- **Political Stability** is the weakest and most volatile resilience domain.
- **Digital Infrastructure** exhibits the strongest long-term improvement between **2000 and 2023**.
- **Healthcare** shows the largest disparity between the highest- and lowest-performing countries.

---

### 🍽️ Food Security

- The **2022 global food price shock** increased commodity prices by approximately **20–40%** compared with **2020**.
- **Oils** and **Cereals** experienced the greatest volatility during the **Russia–Ukraine conflict**.
- Higher food prices appear to be associated with increased undernourishment rates after an approximate **two-year lag**.

---

### ⚠️ Risk Assessment

- Countries with a resilience score below **0.50** are classified as **High Risk**, representing roughly **30–35%** of the sample.
- Countries combining **low Healthcare** performance with **high Undernourishment** are identified as **Double Risk**.
- **Electricity Access** follows a bimodal distribution, with countries tending to have either **near-universal access** or **very limited access**, and relatively few falling in between.